In [22]:
import pandas as pd
import folium
from haversine import haversine

In [23]:
import json

with open('responses.json') as f:
    d = json.load(f)

with open('responses2.json') as f2:
    d2 = json.load(f2)

    

In [24]:
valid = [item for item in d if "scenarioInformation" in item]
valid2 = [item for item in d2 if "scenarioInformation" in item]


In [52]:
def load_in(path):
    with open(path) as f:
        d = json.load(f)
    valid = [item for item in d if "scenarioInformation" in item]
    partial = []
    for item in valid:
        for sub_connection in item["connections"]:
            partial.extend(sub_connection["transportplanElements"])
    normalized_data = pd.json_normalize(partial)
    df = pd.DataFrame(normalized_data)
    reduced = df[["fromInfrastructureLocation.name" , 
              "fromInfrastructureLocation.latitude" ,
              "fromInfrastructureLocation.longitude", 
              "toInfrastructureLocation.name" ,
              "toInfrastructureLocation.latitude", 
              "toInfrastructureLocation.longitude", 
              ]]
    no_dup = reduced.drop_duplicates(["fromInfrastructureLocation.name" , "toInfrastructureLocation.name"]).reset_index(drop="index")
    return no_dup
    
df1 = load_in("responses.json")
df2 = load_in("responses2.json")


In [47]:
# df2.columns = ["toInfrastructureLocation.name" ,
#               "toInfrastructureLocation.latitude", 
#               "toInfrastructureLocation.longitude", 
#               "fromInfrastructureLocation.name" , 
#               "fromInfrastructureLocation.latitude" ,
#               "fromInfrastructureLocation.longitude"
#               ]

In [58]:
combined = pd.concat([df1,df2])
combined = combined.drop_duplicates(["fromInfrastructureLocation.name" , "toInfrastructureLocation.name"])

In [59]:
combined["distance"] = combined.apply(
    lambda row: haversine(
        (row["toInfrastructureLocation.latitude"], row["toInfrastructureLocation.longitude"]),
        (row["fromInfrastructureLocation.latitude"], row["fromInfrastructureLocation.longitude"])
    ),
    axis=1
)

In [67]:
combined

,fromInfrastructureLocation.name,fromInfrastructureLocation.latitude,fromInfrastructureLocation.longitude,toInfrastructureLocation.name,toInfrastructureLocation.latitude,toInfrastructureLocation.longitude,distance
0,Leipzig-Wahren,51.380421,12.322542,Halle (Saale) Gbf,51.483539,11.986633,25.956372
1,Halle (Saale) Gbf,51.483539,11.986633,Seddin,52.291505,12.984401,112.960867
2,Seddin,52.291505,12.984401,Maschen Rbf (Mnof),53.408774,10.052296,232.783866
3,Maschen Rbf (Mnof),53.408774,10.052296,Lüneburg,53.250440,10.419914,30.098671
4,Maschen Rbf (Mnof),53.408774,10.052296,Meckelfeld,53.421844,10.033265,1.924289
...,...,...,...,...,...,...,...
659,Großen Buseck Industriestammgleis,50.598781,8.801221,Wetzlar,50.565703,8.503967,21.307621
660,Ludwigsfelde,52.299155,13.267600,Seddin,52.291505,12.984401,19.277987
661,Duisburg-Wanheim,51.389938,6.749258,DUISBURG-HOCHFELD S,51.408789,6.754565,2.128183
662,Dingolfing PA Scholz,48.641402,12.468090,Dingolfing,48.641622,12.487640,1.436628


In [64]:
h = combined.value_counts("toInfrastructureLocation.name")

In [65]:
h

toInfrastructureLocation.name
Seelze Mitte                 24
NUERNBERG RBF                19
Seddin                       19
GREMBERG                     18
OBERHAUSEN-OSTERFD           18
                             ..
Braunsbedra                   1
Karlsruhe Rheinbrücke AVG     1
Karlsruhe Hbf                 1
Karlsruhe Hafen               1
Kirchheim (Teck) Bosch        1
Name: count, Length: 557, dtype: int64

In [32]:
import plotly.express as px

In [33]:
px.histogram(combined,x = "distance")

In [34]:
m = folium.Map(location=[50, 10], zoom_start=6)
cities = set()
for _, row in directionless.iterrows():
    # Linie zwischen Start- und Zielstation zeichnen
    folium.PolyLine([(row["fromInfrastructureLocation.latitude"], row["fromInfrastructureLocation.longitude"]),
                     (row["toInfrastructureLocation.latitude"], row["toInfrastructureLocation.longitude"])],
                    color="blue", weight=2.5).add_to(m)

    # Start-Station als Marker mit Tooltip
    folium.Marker(
        location=[row["fromInfrastructureLocation.latitude"], row["fromInfrastructureLocation.longitude"]],
        tooltip=row["fromInfrastructureLocation.name"],
        icon=folium.Icon(color="green")
    ).add_to(m)

    # End-Station als Marker mit Tooltip
    folium.Marker(
        location=[row["toInfrastructureLocation.latitude"], row["toInfrastructureLocation.longitude"]],
        tooltip=row["toInfrastructureLocation.name"],
        icon=folium.Icon(color="red")
    ).add_to(m)

# Karte anzeigen
m


In [35]:
max_dis = 0
import math
name = ()
for _, row in reduced.iterrows():
    d = math.dist([row["toInfrastructureLocation.latitude"], row["toInfrastructureLocation.longitude"]] , [row["fromInfrastructureLocation.latitude"], row["fromInfrastructureLocation.longitude"]])
    if d > max_dis:
        max_dis = d
        name = (row["fromInfrastructureLocation.name"] , row["toInfrastructureLocation.name"])

NameError: name 'reduced' is not defined

In [ ]:
name

('Halle (Saale) Gbf', 'KOELN-KALK NORD')

In [ ]:
no_dup

,fromInfrastructureLocation.name,fromInfrastructureLocation.latitude,fromInfrastructureLocation.longitude,toInfrastructureLocation.name,toInfrastructureLocation.latitude,toInfrastructureLocation.longitude
0,Leipzig-Wahren,51.380421,12.322542,Halle (Saale) Gbf,51.483539,11.986633
1,Halle (Saale) Gbf,51.483539,11.986633,Seddin,52.291505,12.984401
2,Seddin,52.291505,12.984401,Maschen Rbf (Mnof),53.408774,10.052296
3,Maschen Rbf (Mnof),53.408774,10.052296,Lüneburg,53.250440,10.419914
4,Maschen Rbf (Mnof),53.408774,10.052296,Meckelfeld,53.421844,10.033265
...,...,...,...,...,...,...
541,Hannover-Linden,52.354079,9.710239,Hildesheim Hbf,52.160644,9.954018
542,Wetzlar,50.565703,8.503967,Großen Buseck Industriestammgleis,50.598781,8.801221
543,Seddin Süd,52.288546,12.983791,Ludwigsfelde,52.299155,13.267600
544,DUISBURG-HOCHFELD S,51.408789,6.754565,Duisburg-Wanheim,51.389938,6.749258


In [ ]:
df.columns

Index(['blockName', 'compositionEndWeekdays', 'compositionEndTimeOfDay',
       'departureWeekdays', 'departureTimeOfDay', 'arrivalWeekdays',
       'arrivalTimeOfDay', 'decompositionEndWeekdays',
       'decompositionEndTimeOfDay', 'runDuration', 'runLength',
       'trainsetLength', 'trainsetWeight', 'maxVelocity', 'trainNumber',
       'combinedTrafficCoding', 'linkClass', 'trainCategory',
       'fromInfrastructureLocation.code',
       'fromInfrastructureLocation.countryNumber',
       'fromInfrastructureLocation.locationNumber',
       'fromInfrastructureLocation.freightCarLocationNumber',
       'fromInfrastructureLocation.name',
       'fromInfrastructureLocation.latitude',
       'fromInfrastructureLocation.longitude',
       'fromInfrastructureLocation.objectKeyAlpha',
       'fromInfrastructureLocation.objectKeySequence',
       'fromInfrastructureLocation.tafTsiPrimaryCode',
       'fromInfrastructureLocation.tafTsiSubsidiaryType',
       'toInfrastructureLocation.code',
  

In [ ]:
with open("test.json", "w") as f:
    f.write(json.dumps(test))

In [ ]:
l = df["transportplanElements"]

KeyError: 'transportplanElements'

In [ ]:
for i in l:
    if len(i) > 1:
        print(len(i))

4
4
4
6
6
4
4
5
5
6
6
5
5
5
5
4
4
4
3
5
5
4
5
5
5
5
4
4
5
5
5
5
5
5
5
5
4
4
5
5
4
4
2
3
3
4
4
6
6
6
6
4
4
4
4
4
4
4
4
4
4
5
4
4
4
4
5
4
6
5
6
6
6
6
5
5
5
5
5
5
4
4
4
4
3
3
3
3
4
4
4
4
3
3
4
4
3
3
4
4
4
4
6
5
5
4
4
4
4
5
5
3
3
3
3
4
4
5
5
5
5
3
3
4
4
4
4
5
5
4
4
5
5
5
5
4
4
5
5
5
5
5
5
4
4
5
5
5
5
5
5
6
6
5
5
5
5
3
6
6
5
5
5
5
6
6
5
5
5
5
5
5
4
4
6
6
4
4
4
4
4
4
4
4
5
5
4
4
5
5
6
6
7
7
4
4
4
4
4
4
4
4
6
6
4
4
4
4
4
4
4
4
4
4
4
4
5
5
5
4
3
3
4
4
3
3
3
3
3
6
6
4
4
4
4
5
5
2
3
5
5
5
5
4
4
4
4
5
5
3
3
5
5
4
4
3
3
3
3
3
3
4
4
3
3
3
3
3
4
4
3
3
5
5
2
5
5
7
7
3
3
3
3
3
3
3
3
3
3
3
5
5
5
5
5
5
5
5
4
4
4
4
4
4
6
6
3
3
5
5
6
6
5
5
7
7
3
3
4
4
5
5
5
4
4
6
6
5
5
5
5
3
5
5
5
5
5
5
5
5
4
4
3
3
5
5
3
4
4
4
4
6
6
3
3
5
5
5
5
4
4
4
4
4
3
3
3
3
4
4
3
3
3
3
4
4
4
4
3
3
4
4
5
5
5
5
6
6
2
3
3
2
2
2
3
4
4
3
3
4
4
4
4
2
4
4
4
4
5
4
5
5
5
5
5
5
4
4
4
4
3
3
4
4
5
5
5
5
3
4
4
4
4
4
4
4
4
3
4
4
4
4
4
4
4
4
5
4
5
5
5
5
4
5
5
4
4
6
6
4
4
4
4
4
4
5
5
4
5
5
4
3
3
4
4
5
5
5
5
4
4
4
4
4
4
5
5
5
5
5
5
6
6
6
5
5
7
7
5
5


In [ ]:
len(connections)

828

In [ ]:
for item in d:
    print("scenarioInformation" in item)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
True
True
True
True
True
True
True
True
False
True
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
False
True
False
True
False
False
False
False
False
False
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True
False
False
True
True
False
False
False
False
False
False
False
False
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
